# Decision Tree Learning – Algorithm 4 (Exam Bonus Problem)

### Group Members:
- Name 1, Student ID 1  
- Name 2, Student ID 2  
- Name 3, Student ID 3  
- Name 4, Student ID 4  

---
## Step 1: Prepare the Data
We'll use a version of the classic "Play Tennis" dataset, converted to use only integer (binary/categorical) attributes.

In [1]:
import pandas as pd

data = [
    # Outlook, Temperature, Humidity, Windy, PlayTennis
    ['Sunny', 'Hot', 'High', False, 'No'],
    ['Sunny', 'Hot', 'High', True, 'No'],
    ['Overcast', 'Hot', 'High', False, 'Yes'],
    ['Rain', 'Mild', 'High', False, 'Yes'],
    ['Rain', 'Cool', 'Normal', False, 'Yes'],
    ['Rain', 'Cool', 'Normal', True, 'No'],
    ['Overcast', 'Cool', 'Normal', True, 'Yes'],
    ['Sunny', 'Mild', 'High', False, 'No'],
    ['Sunny', 'Cool', 'Normal', False, 'Yes'],
    ['Rain', 'Mild', 'Normal', False, 'Yes'],
    ['Sunny', 'Mild', 'Normal', True, 'Yes'],
    ['Overcast', 'Mild', 'High', True, 'Yes'],
    ['Overcast', 'Hot', 'Normal', False, 'Yes'],
    ['Rain', 'Mild', 'High', True, 'No']
]

df = pd.DataFrame(data, columns=['Outlook', 'Temperature', 'Humidity', 'Windy', 'PlayTennis'])

# Convert to numeric/categorical codes
df['Outlook'] = df['Outlook'].map({'Sunny': 0, 'Overcast': 1, 'Rain': 2})
df['Temperature'] = df['Temperature'].map({'Hot': 0, 'Mild': 1, 'Cool': 2})
df['Humidity'] = df['Humidity'].map({'High': 0, 'Normal': 1})
df['Windy'] = df['Windy'].astype(int)
df['PlayTennis'] = df['PlayTennis'].map({'No': 0, 'Yes': 1})
df

,Outlook,Temperature,Humidity,Windy,PlayTennis
0,0,0,0,0,0
1,0,0,0,1,0
2,1,0,0,0,1
3,2,1,0,0,1
4,2,2,1,0,1
5,2,2,1,1,0
6,1,2,1,1,1
7,0,1,0,0,0
8,0,2,1,0,1
9,2,1,1,0,1


## Step 2: Helper Functions
### Plurality Value
This function returns the most common class label.

In [2]:
def plurality_value(examples, target_attribute):
    return examples[target_attribute].mode()[0]

### Entropy
Entropy measures the impurity of a dataset with respect to the target attribute.

In [3]:
import numpy as np
def entropy(examples, target_attribute):
    values, counts = np.unique(examples[target_attribute], return_counts=True)
    probabilities = counts / counts.sum()
    return -np.sum(probabilities * np.log2(probabilities + 1e-9))

### Information Gain
Information gain tells us which attribute splits the data best.

In [4]:
def information_gain(examples, attribute, target_attribute):
    total_entropy = entropy(examples, target_attribute)
    values = examples[attribute].unique()
    weighted_entropy = 0
    for v in values:
        subset = examples[examples[attribute] == v]
        weighted_entropy += (len(subset) / len(examples)) * entropy(subset, target_attribute)
    return total_entropy - weighted_entropy

## Step 3: Decision Tree Node Class
This class represents each node (attribute split or leaf) in the tree.

In [5]:
class TreeNode:
    def __init__(self, attribute=None, is_leaf=False, classification=None):
        self.attribute = attribute
        self.is_leaf = is_leaf
        self.classification = classification
        self.children = {}  # value: TreeNode

    def add_child(self, value, node):
        self.children[value] = node

## Step 4: Implement Algorithm 4 Recursively
This is the core of the algorithm, directly following the pseudocode.

In [6]:
def dt_learning(examples, attributes, parent_examples, target_attribute):
    if len(examples) == 0:
        return TreeNode(is_leaf=True, classification=plurality_value(parent_examples, target_attribute))
    elif len(examples[target_attribute].unique()) == 1:
        return TreeNode(is_leaf=True, classification=examples[target_attribute].iloc[0])
    elif len(attributes) == 0:
        return TreeNode(is_leaf=True, classification=plurality_value(examples, target_attribute))
    else:
        # Compute information gain for each attribute
        gains = {a: information_gain(examples, a, target_attribute) for a in attributes}
        A = max(gains, key=gains.get)
        node = TreeNode(attribute=A)
        for v in sorted(examples[A].unique()):
            subset = examples[examples[A] == v]
            remaining_attributes = [attr for attr in attributes if attr != A]
            child = dt_learning(subset, remaining_attributes, examples, target_attribute)
            node.add_child(v, child)
        return node

## Step 5: Train the Tree

In [7]:
attributes = ['Outlook', 'Temperature', 'Humidity', 'Windy']
target_attribute = 'PlayTennis'
tree = dt_learning(df, attributes, df, target_attribute)

## Step 6: Visualize the Learned Tree
We use `graphviz` to draw the tree. You may need to install graphviz with `pip install graphviz` and install the Graphviz system package for visualization to work.

In [8]:
from graphviz import Digraph

def render_tree(node, dot=None, parent=None, edge_label=''):
    if dot is None:
        dot = Digraph()
    node_id = str(id(node))
    if node.is_leaf:
        dot.node(node_id, f"Leaf: {node.classification}")
    else:
        dot.node(node_id, f"{node.attribute}")
    if parent is not None:
        dot.edge(parent, node_id, label=str(edge_label))
    for attr_value, child in node.children.items():
        render_tree(child, dot, node_id, edge_label=attr_value)
    return dot

dot = render_tree(tree)
dot.render('decision_tree', format='png', view=True)  # view=True will open the image in a viewer

'decision_tree.png'

## Step 7: Predict with the Tree
Here's how to use the tree for prediction:

In [9]:
def predict(tree, instance):
    node = tree
    while not node.is_leaf:
        attr = node.attribute
        v = instance[attr]
        if v in node.children:
            node = node.children[v]
        else:
            return None  # Unknown branch
    return node.classification

# Example prediction
sample = {'Outlook': 0, 'Temperature': 1, 'Humidity': 0, 'Windy': 1}
print("Prediction (PlayTennis):", predict(tree, sample))

Prediction (PlayTennis): 0


Gtk-Message: 21:27:39.807: Failed to load module "xapp-gtk3-module"
Gtk-Message: 21:27:39.807: Failed to load module "canberra-gtk-module"
[0621/212739.835103:WARNING:chrome/app/chrome_main_linux.cc:82] Read channel stable from /app/extra/CHROME_VERSION_EXTRA


[0621/212739.941409:WARNING:chrome/app/chrome_main_linux.cc:82] Read channel stable from /app/extra/CHROME_VERSION_EXTRA
Opening in existing browser session.


## Step 8: Summary
- We implemented Algorithm 4 for decision tree learning with binary/categorical attributes.
- The tree is learned from the data and visualized.
- No scikit-learn or similar libraries are used for the learning part.
- Add your names and IDs at the top before submitting!

**Export this notebook as HTML and submit as required.**